# Website Textextraction Selenium — Google Colab API

FastAPI service for crawling web pages and converting them to Markdown,
with JavaScript rendering via Selenium, PII anonymisation, and Cloudflare Tunnel
for public access.

**Repository:** https://github.com/janschachtschabel/Website-Textextraction-Selenium

---

## 🚀 Key Features

| Feature | Details |
|---|---|
| **Auto Mode** | Preflight probe → HTTP-only or Selenium as needed |
| **JS Rendering** | Selenium + Chrome with stealth mode, cookie-banner handling |
| **File Formats** | PDF, DOCX, PPTX, XLSX and 20+ formats via MarkItDown |
| **Converters** | `trafilatura` (default) or `markitdown` |
| **Screenshots** | Base64 viewport PNG (Selenium path only) |
| **Link Extraction** | Categorised: content / nav / social / legal / auth / search / contact / download |
| **PII Anonymisation** | Local Presidio (de + en), no external API key required |
| **Caching** | Shared diskcache, 30 min TTL, 500 MB limit |
| **word_count** | Word count of extracted Markdown |
| **error_page_detected** | Detects 4xx/5xx, captcha pages, and thin/empty content |

---

## 📊 Request Schema (`POST /crawl`)

```json
{
  "url": "https://example.com",
  "mode": "auto",
  "screenshot": false,
  "html_converter": "trafilatura",
  "trafilatura_clean_markdown": true,
  "extract_links": false,
  "anonymize": false,
  "anonymize_language": "de",
  "force_refresh": false,
  "timeout_ms": 60000,
  "js_strategy": "speed",
  "retries": 1,
  "max_bytes": 10485760,
  "crawl_rate_limit_rps": 0
}
```

## 📦 Response Schema

```json
{
  "request_mode": "auto",
  "requested_url": "https://example.com",
  "final_url": "https://example.com",
  "status_code": 200,
  "redirected": false,
  "content_type": "text/html; charset=utf-8",
  "markdown": "# Example Domain\n...",
  "markdown_length": 1234,
  "word_count": 210,
  "error_page_detected": false,
  "links": null,
  "screenshot_base64": null,
  "elapsed_ms": 850,
  "cached": false,
  "anonymization": null
}
```

---

## 🎯 Crawl Modes

| Mode | Path | Speed | Use case |
|---|---|---|---|
| `auto` | Preflight → decides | ~0.5 s (HTTP) / ~3–6 s (JS) | General use — recommended |
| `fast` | Plain HTTP only | ~0.5–1 s | Known static sites |
| `js` | Selenium always | ~3–6 s | Known JS-heavy / SPA sites |

---

## 🔗 API Endpoints

| Endpoint | Description |
|---|---|
| `POST /crawl` | Single URL extraction |
| `POST /crawl/batch` | Batch extraction |
| `GET /health` | Server + Selenium pool status |
| `GET /stats` | Operational metrics |
| `GET /docs` | Swagger UI |

---

## ⚙️ Colab Resource Profile (Performance Setting)

- ~12 GB RAM, 2 vCPUs
- **2 uvicorn workers** · Selenium pool **2→4 per worker** (max 8 Chrome)
- Sweet-spot concurrency: **4 parallel requests** at ~3 s average latency


In [ ]:
# ======================================================================================
# Google Colab Deployment — Website Textextraction Selenium API
# Repository: https://github.com/janschachtschabel/Website-Textextraction-Selenium
#
# Colab resources: ~12 GB RAM, 2 vCPUs
# Configuration: Performance Setting — 2 workers, Selenium pool 2→4, no LLM (Presidio local)
# ======================================================================================

print("🚀 Website Textextraction Selenium — Google Colab Deployment")
print("=" * 60)

import functools
import os
import re
import subprocess
import sys
import threading
import time

import requests

# Set environment variables
# Performance Setting: 2 workers, pool 2→4, cache 30 min
# For low-resource runtimes use Safe Setting instead:
#   UVICORN_WORKERS=1, SELENIUM_POOL_SIZE=1, SELENIUM_MAX_POOL_SIZE=4,
#   RESULT_CACHE_TTL=300, RESULT_CACHE_MAX_SIZE=200
os.environ.update({
    "HOST":                      "0.0.0.0",
    "PORT":                      "8000",
    "LOG_LEVEL":                 "INFO",
    "DEFAULT_MODE":              "auto",
    "DEFAULT_JS_STRATEGY":       "speed",
    "DEFAULT_TIMEOUT_SECONDS":   "120",
    "DEFAULT_RETRIES":           "1",
    "DEFAULT_HEADLESS":          "true",
    "DEFAULT_STEALTH":           "true",
    "DEFAULT_JS_AUTO_WAIT":      "true",
    "DEFAULT_MAX_BYTES":         "10485760",
    # Performance Setting: 2 workers × 4 Chrome = 8 parallel JS renders
    # Load-test result: sweet-spot conc=4 at ~3 s latency, 0 errors
    "SELENIUM_POOL_SIZE":        "2",   # 4 Chrome at startup (2 per worker)
    "SELENIUM_MAX_POOL_SIZE":    "4",   # max. 8 Chrome (4 per worker)
    "SELENIUM_SCALE_THRESHOLD":  "0.8",
    "UVICORN_WORKERS":           "2",   # 2 workers; each warms its pool independently
    "MAX_QUEUE_SIZE":            "30",
    "QUEUE_TIMEOUT_SECONDS":     "90",
    "HTML_CONVERTER":            "trafilatura",
    "TRAFILATURA_CLEAN_MARKDOWN":"true",
    "RESULT_CACHE_TTL":          "1800", # 30 min – Colab content rarely changes
    "RESULT_CACHE_MAX_SIZE":     "500",  # 500 MB
    "ALLOW_INSECURE_SSL":        "false",
    "SSRF_PROTECTION":           "true",
    "GLOBAL_RATE_LIMIT_RPS":     "0",
    "DEFAULT_DOMAIN_RATE_LIMIT_RPS": "0",
    "MEDIA_CONVERSION_POLICY":   "skip",
    "CHROME_BINARY":              "/usr/bin/google-chrome",
})

# Install system packages
print("📦 Installing system packages...")
subprocess.run("sudo apt-get update -qq && sudo apt-get install -y git", shell=True, check=True)

# Install Chrome — skip if already present, otherwise download .deb directly
print("🌐 Installing Google Chrome...")
_chrome_present = subprocess.run(
    ["which", "google-chrome-stable"], capture_output=True
).returncode == 0 or subprocess.run(
    ["which", "google-chrome"], capture_output=True
).returncode == 0

if _chrome_present:
    print("✅ Google Chrome already installed — skipping")
else:
    subprocess.run(
        "wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"
        " -O /tmp/google-chrome-stable.deb",
        shell=True, check=True,
    )
    subprocess.run("sudo dpkg -i /tmp/google-chrome-stable.deb", shell=True)
    subprocess.run("sudo apt-get install -f -y -q", shell=True, check=True)
    print("✅ Google Chrome installed")

# Install Cloudflare Tunnel
print("🌐 Installing Cloudflare Tunnel...")
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb", shell=True, check=True)
subprocess.run("sudo dpkg -i cloudflared-linux-amd64.deb", shell=True, check=True)

# Clone repository
print("📥 Cloning repository...")
subprocess.run("git clone https://github.com/janschachtschabel/Website-Textextraction-Selenium.git /content/Website-Textextraction-Selenium", shell=True, check=True)
os.chdir('/content/Website-Textextraction-Selenium')
sys.path.insert(0, '/content/Website-Textextraction-Selenium')

# Install Python dependencies (explicit list — no pyproject.toml required)
print("📦 Installing Python dependencies...")
_PACKAGES = [
    "fastapi[standard]>=0.135.1",
    "uvicorn[standard]>=0.42.0",
    "httpx[http2]>=0.28.1",
    "selenium>=4.41.0",
    "webdriver-manager>=4.0.2",
    "markitdown[all]>=0.1.5",
    "trafilatura>=2.0.0",
    "beautifulsoup4>=4.14.3",
    "lxml>=5.3.0",
    "python-dotenv>=1.0.1",
    "tqdm>=4.66.0",
    "cachetools>=7.0.0",
    "loguru>=0.7.3",
    "truststore>=0.10.0",
    "aiolimiter>=1.1.0",
    "diskcache>=5.6.0",
    "presidio-analyzer>=2.2.0",
    "presidio-anonymizer>=2.2.0",
    "spacy>=3.7.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _PACKAGES, check=True)
print("✅ Python dependencies installed")

# Download spaCy models for Presidio anonymisation
print("🧠 Downloading spaCy models for PII anonymisation (one-time ~500 MB)...")
subprocess.run([sys.executable, "-m", "spacy", "download", "de_core_news_lg", "-q"], check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_lg", "-q"], check=True)
print("✅ spaCy models loaded")

# =============================================================================
# CHROME PATCH — set binary_location only
# All other flags (--no-sandbox, --disable-dev-shm-usage, etc.) are already
# included in js_fetcher._create_driver and must not be set again.
# =============================================================================

def _patch_chrome_for_colab():
    """Injects binary_location into every Options call made by js_fetcher."""
    try:
        from selenium.webdriver.chrome.options import Options
        _orig_init = Options.__init__

        @functools.wraps(_orig_init)
        def _patched_init(self, *args, **kwargs):
            _orig_init(self, *args, **kwargs)
            self.binary_location = "/usr/bin/google-chrome"

        Options.__init__ = _patched_init
        print("✅ Chrome binary_location set for Colab (/usr/bin/google-chrome)")
    except Exception as e:
        print(f"⚠️ Chrome patch failed: {e}")

os.environ["PYTHONPATH"] = "/content/Website-Textextraction-Selenium"
os.chdir("/content/Website-Textextraction-Selenium")

# =============================================================================
# HEALTH CHECK FUNCTIONS
# =============================================================================

def check_fastapi_health(port=8000, max_attempts=40):
    """Waits until the API is ready. Accepts /health (200) or falls back to /docs (200)."""
    for attempt in range(max_attempts):
        try:
            response = requests.get(f"http://localhost:{port}/health", timeout=3)
            if response.status_code == 200:
                data = response.json()
                if data.get("status") in ("ok", "starting"):
                    pools_ready = not data.get("pools_warming", True)
                    if pools_ready:
                        print(f"✅ API ready — Selenium pools initialised (attempt {attempt + 1})")
                        return True
                    else:
                        print(f"⏳ API running, Selenium pools still warming... (attempt {attempt + 1}/{max_attempts})")
                        time.sleep(3)
                        continue
                else:
                    # Unexpected 200 body — still treat as up
                    print(f"✅ API responding (attempt {attempt + 1}) — body: {str(data)[:120]}")
                    return True
            elif response.status_code == 404:
                # /health route not found — old repo version; confirm via /docs
                docs = requests.get(f"http://localhost:{port}/docs", timeout=3)
                if docs.status_code == 200:
                    print(f"⚠️  /health returned 404 (repo may be outdated) but API is up — continuing")
                    return True
        except requests.exceptions.RequestException:
            pass

        print(f"⏳ Waiting for API to start... (attempt {attempt + 1}/{max_attempts})")
        time.sleep(3)

    return False

def run_fastapi():
    """Starts FastAPI (2 workers, no --reload in Colab)."""
    print("🚀 Starting API server (Performance Setting: 2 workers, pool 2→4)...")
    os.chdir("/content/Website-Textextraction-Selenium")

    process = subprocess.Popen(
        ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    # Stream ALL output to cell in a background thread — never break the loop.
    # A break would stop draining the pipe; the 64 KB OS buffer fills up;
    # workers block on stdout writes and stop responding to HTTP requests.
    def _stream():
        for line in process.stdout:
            print(line.rstrip(), flush=True)
    threading.Thread(target=_stream, daemon=True).start()

def start_cloudflare_tunnel(port):
    """Starts Cloudflare Tunnel and extracts the public URL."""
    print(f"🌐 Starting Cloudflare Tunnel for port {port}...")

    process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    for line in process.stderr:
        print(f"Cloudflare: {line.strip()}")
        if "trycloudflare.com" in line:
            match = re.search(r'https?://[^\s]+', line)
            if match:
                return match.group(0)

    return None

# =============================================================================
# MAIN DEPLOYMENT SCRIPT
# =============================================================================

# Step 1: start FastAPI in a separate thread
print("🔧 Starting API server (Performance Setting: 2 workers, Selenium pool 2→4)...")
fastapi_thread = threading.Thread(target=run_fastapi)
fastapi_thread.daemon = True
fastapi_thread.start()

# Step 2: wait and health-check (Selenium pool needs ~15–30 s)
print("⏳ Waiting for API + Selenium pool...")
time.sleep(20)

if check_fastapi_health():
    print("✅ Website-Textextraction-Selenium API is running!")

    # Step 3: start Cloudflare Tunnel
    tunnel_url = start_cloudflare_tunnel(8000)

    if tunnel_url:
        with open("/tmp/tunnel_url.txt", "w") as _f:
            _f.write(tunnel_url)
        print("\n🎉 API available at:")
        print(f"🌐 Public URL:      {tunnel_url}")
        print(f"📚 API Docs:        {tunnel_url}/docs")
        print(f"❤️  Health:          {tunnel_url}/health")
        print(f"📊 Stats:           {tunnel_url}/stats")
        print(f"🔗 Crawl Endpoint:  {tunnel_url}/crawl")

        print("\n📋 Example requests:")
        print(f"""
# Simple crawl (auto mode, with screenshot)
curl -X POST "{tunnel_url}/crawl" \\
  -H "Content-Type: application/json" \\
  -d '{{"url": "https://example.com", "mode": "auto", "screenshot": true}}'

# PII anonymisation (local, no API key)
curl -X POST "{tunnel_url}/crawl" \\
  -H "Content-Type: application/json" \\
  -d '{{"url": "https://example.com", "anonymize": true, "anonymize_language": "en"}}'

# Set per-domain rate limit
curl -X POST "{tunnel_url}/crawl" \\
  -H "Content-Type: application/json" \\
  -d '{{"url": "https://example.com", "crawl_rate_limit_rps": 0.5}}'
        """)

    else:
        print("❌ Cloudflare Tunnel could not be started")

else:
    print("❌ Website-Textextraction-Selenium API failed to start!")
    print("🔍 Debugging information:")

    # Directory check
    print("\n📁 Directory check:")
    subprocess.run(["ls", "-la", "/content/Website-Textextraction-Selenium/"], check=False)

    print("\n📁 App module check:")
    subprocess.run(["ls", "-la", "/content/Website-Textextraction-Selenium/app/"], check=False)

    # Dependencies check
    print("\n📦 Dependencies check:")
    subprocess.run([sys.executable, "-m", "pip", "show", "fastapi", "uvicorn", "selenium", "markitdown", "presidio-analyzer"], check=False)

    # App import test
    print("\n🔧 App import test:")
    subprocess.run([sys.executable, "-c", "from app.main import app; print('\u2705 App import successful')"], check=False, cwd="/content/Website-Textextraction-Selenium")

print("\n" + "=" * 60)
print("🏁 Deployment script finished")


In [ ]:
# =============================================================================
# Running processes — inspect API server, Chrome instances, Cloudflare Tunnel
# Run this cell at any time to see what is alive in the Colab runtime.
# =============================================================================

import subprocess

result = subprocess.run(["ps", "aux", "--no-headers"], capture_output=True, text=True)
lines = result.stdout.strip().split("\n")

keywords = ["uvicorn", "python", "chrome", "google-chrome", "cloudflared"]
matches = [l for l in lines if any(k in l.lower() for k in keywords)]

print(f"{'PID':>7}  {'%CPU':>5}  {'%MEM':>5}  COMMAND")
print("-" * 80)
for line in sorted(matches, key=lambda l: float(l.split()[2]) if len(l.split()) > 2 else 0, reverse=True)[:30]:
    parts = line.split(None, 10)
    if len(parts) >= 11:
        pid, cpu, mem, cmd = parts[1], parts[2], parts[3], parts[10][:60]
        print(f"{pid:>7}  {cpu:>5}  {mem:>5}  {cmd}")
    elif len(parts) >= 4:
        print(f"{'?':>7}  {parts[2]:>5}  {parts[3]:>5}  {' '.join(parts[4:])[:60]}")

print(f"\nTotal matching processes: {len(matches)}")


In [ ]:
# =============================================================================
# API health & statistics — queries localhost:8000 directly (no tunnel needed)
# Run this cell after the deployment cell has finished successfully.
# =============================================================================

import requests, json, pathlib

BASE = "http://localhost:8000"

# Show public tunnel URL if available
_url_file = pathlib.Path("/tmp/tunnel_url.txt")
if _url_file.exists():
    print(f"🌐 Public tunnel URL: {_url_file.read_text().strip()}")
print()

# ── Health ────────────────────────────────────────────────────────────────────
print("=== /health ===")
try:
    h = requests.get(f"{BASE}/health", timeout=10).json()
    print(json.dumps(h, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"  ❌ {e}")

print()

# ── Stats ─────────────────────────────────────────────────────────────────────
print("=== /stats ===")
try:
    s = requests.get(f"{BASE}/stats", timeout=10).json()

    print(f"  concurrent_requests : {s.get('concurrent_requests')}")
    print(f"  max_concurrent      : {s.get('max_concurrent')}")
    print(f"  queue_size          : {s.get('queue_size')}")
    print(f"  max_queue_size      : {s.get('max_queue_size')}")

    pools = s.get("selenium_pools") or {}
    if pools:
        print("\n  selenium_pools:")
        for strategy, info in pools.items():
            print(f"    {strategy}: {json.dumps(info)}")

    lh = s.get("last_hour") or {}
    if lh:
        print("\n  last_hour:")
        print(f"    requests_total   : {lh.get('requests_total', 0)}")
        print(f"    requests_success : {lh.get('requests_success', 0)}")
        print(f"    requests_error   : {lh.get('requests_error', 0)}")
        print(f"    cache_hits       : {lh.get('cache_hits', 0)}")
        print(f"    cache_hit_rate   : {lh.get('cache_hit_rate_pct', 0):.1f} %")

        lat = lh.get("latency_seconds") or {}
        if lat:
            print("\n    latency (seconds):")
            for mode, v in lat.items():
                if isinstance(v, dict) and v.get("n", 0) > 0:
                    print(f"      {mode:6s}  p50={v.get('p50',0):.2f}  p95={v.get('p95',0):.2f}  avg={v.get('avg',0):.2f}  n={v.get('n')}")
except Exception as e:
    print(f"  ❌ {e}")


In [ ]:
# =============================================================================
# Demo crawl requests — uses localhost:8000 directly (no tunnel round-trip).
# The public tunnel URL is only needed for access from outside Colab.
# =============================================================================

import requests, json, textwrap, pathlib

BASE = "http://localhost:8000"
CRAWL = BASE + "/crawl"

# Show the public tunnel URL for reference
_url_file = pathlib.Path("/tmp/tunnel_url.txt")
if _url_file.exists():
    print(f"🌐 Public tunnel URL: {_url_file.read_text().strip()}")
    print("   (Use this URL to access the API from outside Colab)")
    print()

def crawl(url, **kwargs):
    payload = {"url": url, "mode": "auto", "screenshot": False, **kwargs}
    r = requests.post(CRAWL, json=payload, timeout=90)
    r.raise_for_status()
    return r.json()

def show(d):
    print(f"  status          : {d.get('status_code')}  ({d.get('request_mode')} mode)")
    print(f"  final_url       : {d.get('final_url')}")
    print(f"  word_count      : {d.get('word_count')}")
    print(f"  markdown_length : {d.get('markdown_length')}")
    print(f"  error_detected  : {d.get('error_page_detected')}")
    print(f"  elapsed_ms      : {d.get('elapsed_ms')}")
    md = d.get("markdown", "")
    if md:
        print("  preview         :", textwrap.shorten(md, 120))
    print()

# ── Static site ───────────────────────────────────────────────────────────────
print("=== Static site — auto mode ===")
show(crawl("https://example.com"))

# ── JS-heavy site ─────────────────────────────────────────────────────────────
print("=== JS-heavy site — auto mode ===")
show(crawl("https://codeweek.eu/"))

# ── Link extraction ───────────────────────────────────────────────────────────
print("=== Link extraction ===")
d = crawl("https://example.com", extract_links=True)
show(d)
links = d.get("links") or []
for lnk in links[:10]:
    print(f"  [{lnk.get('category'):10s}] {'INT' if lnk.get('internal') else 'EXT'}  {lnk.get('url')}")
print()

# ── Batch crawl ───────────────────────────────────────────────────────────────
print("=== Batch crawl ===")
batch_payload = {
    "urls": [
        "https://example.com",
        "https://www.wikipedia.org",
        "https://computingeducation.de/",
    ],
    "mode": "auto",
    "screenshot": False,
}
resp = requests.post(BASE + "/crawl/batch", json=batch_payload, timeout=120)
resp.raise_for_status()
for item in resp.json().get("results", []):
    r = item.get("result") or {}
    ok = item.get("success")
    print(f"  {'✅' if ok else '❌'}  {item.get('url')[:60]:<60}  words={r.get('word_count', '?')}")
